# Test I: Audio Transcription
## AutoEIT GSoC 2026 Evaluation
Transcription of Spanish EIT audio recordings using OpenAI Whisper.

In [24]:
import whisper
import pandas as pd
import openpyxl
from openpyxl import load_workbook
import os
import re
import subprocess
import numpy as np

# Add ffmpeg to PATH
os.environ["PATH"] += r";C:\ffmpeg\ffmpeg-master-latest-win64-gpl-shared\bin"

# Paths
AUDIO_DIR = r"C:\Users\kashi\OneDrive\Desktop\AutoEIT datasets\AutoEIT Test Files\Sample Audio Files and Transcriptions"
EXCEL_PATH = r"C:\Users\kashi\OneDrive\Desktop\AutoEIT datasets\AutoEIT Test Files\Sample Audio Files and Transcriptions\AutoEIT Sample Audio for Transcribing.xlsx"
OUTPUT_PATH = r"C:\Users\kashi\OneDrive\Desktop\AutoEIT\autoeit\output\test1_output.xlsx"

PARTICIPANTS = {
    "38010-2A": "038010_EIT-2A.mp3",
    "38011-1A": "038011_EIT-1A.mp3",
    "38012-2A": "038012_EIT-2A.mp3",
    "38015-1A": "038015_EIT-1A.mp3",
}

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
print("Setup complete")

Setup complete


In [25]:
print("Loading Whisper base model...")
model = whisper.load_model("base")
print("Model loaded!")

Loading Whisper base model...
Model loaded!


In [27]:
def transcribe_audio(audio_path):
    """
    Transcribe audio file using Whisper.
    Skips first 150 seconds (English instructions + practice sentences).
    Preserves exact learner production including disfluencies.
    """
    print(f"Transcribing {os.path.basename(audio_path)}...")
    
    # Load audio and skip first 150 seconds (2:30)
    audio = whisper.load_audio(audio_path)
    start_sample = 150 * 16000  # 150s * 16000 sample rate
    audio = audio[start_sample:]
    
    result = model.transcribe(
        audio,
        language="es",
        verbose=False,
        initial_prompt="Elicited imitation task in Spanish."
    )
    
    segments = [seg["text"].strip() for seg in result["segments"]]
    segments = [s for s in segments if s.strip()]
    print(f"  Got {len(segments)} segments")
    return segments

In [28]:
all_transcriptions = {}

for sheet_name, audio_file in PARTICIPANTS.items():
    audio_path = os.path.join(AUDIO_DIR, audio_file)
    all_transcriptions[sheet_name] = transcribe_audio(audio_path)

print("\nDone transcribing all files!")

Transcribing 038010_EIT-2A.mp3...


100%|██████████| 39963/39963 [02:29<00:00, 267.97frames/s]


  Got 25 segments
Transcribing 038011_EIT-1A.mp3...


 95%|█████████▍| 37700/39703 [03:26<00:10, 182.25frames/s]


  Got 23 segments
Transcribing 038012_EIT-2A.mp3...


100%|██████████| 97554/97554 [03:12<00:00, 508.02frames/s]


  Got 48 segments
Transcribing 038015_EIT-1A.mp3...


100%|██████████| 37937/37937 [02:52<00:00, 220.13frames/s]

  Got 23 segments

Done transcribing all files!


In [29]:
wb = load_workbook(EXCEL_PATH)

for sheet_name, segments in all_transcriptions.items():
    print(f"Writing {sheet_name}... ({len(segments)} segments)")
    ws = wb[sheet_name]
    
    for row_num in range(2, 32):  # 30 stimulus rows
        seg_idx = row_num - 2
        if seg_idx < len(segments):
            ws.cell(row=row_num, column=3, value=segments[seg_idx])
        else:
            ws.cell(row=row_num, column=3, value="[no transcription]")

wb.save(OUTPUT_PATH)
print(f"\nSaved to {OUTPUT_PATH}")

Writing 38010-2A... (25 segments)
Writing 38011-1A... (23 segments)
Writing 38012-2A... (48 segments)
Writing 38015-1A... (23 segments)

Saved to C:\Users\kashi\OneDrive\Desktop\AutoEIT\autoeit\output\test1_output.xlsx


In [30]:
wb_out = load_workbook(OUTPUT_PATH)
ws = wb_out["38010-2A"]

print("Sample output - first 5 rows (38010-2A):\n")
for row_num in range(2, 7):
    stimulus = ws.cell(row=row_num, column=2).value
    transcription = ws.cell(row=row_num, column=3).value
    print(f"S: {stimulus}")
    print(f"T: {transcription}")
    print()

Sample output - first 5 rows (38010-2A):

S: Quiero cortarme el pelo (7)
T: Quiero cortarme al pelo.

S: El libro está en la mesa (7)
T: El libro está en la mesa.

S: El carro lo tiene Pedro (8)
T: El caro lo tiene pelo.

S: El se ducha cada mañana (9)
T: El se dice que es caramagnana.

S: ¿Qué dice usted que va a hacer hoy? (9)
T: El libro está en la mesa.



## Methodology & Approach

### Tool Selection
OpenAI Whisper (base model) was selected for ASR due to its strong 
multilingual support and open-source availability. The medium/large 
models would yield better accuracy in production but are too slow for 
CPU-only environments.

### Key Decisions
- Language explicitly set to Spanish (`language="es"`)
- First 150 seconds skipped as the recording contains English instructions
- Transcriptions preserve exact learner production including disfluencies
- ASR errors corrected, but learner grammar/vocabulary errors preserved

### Known Limitations
- Whisper's VAD occasionally merges consecutive responses due to long 
  pauses in EIT recordings (participant listening time)
- Some responses were missed or merged, resulting in fewer than 30 segments 
  for some participants
- A production system would benefit from forced alignment tools like 
  WhisperX or stable-ts for precise sentence-level alignment

### Evaluation of Output
Output was evaluated by comparing Whisper transcriptions against the 
provided stimulus sentences, checking for semantic preservation and 
Spanish word recognition accuracy.